# Milestone 4 — Best model & error analysis

Same split, same target (`diagnosis_1`, 3 classes), same metric (macro-F1) as Milestones 2 and 3.

1. ResNet-18 vs ViT-B/16, both frozen, same head (Session 9's fair comparison)
2. The best model: pick on validation, use the test set once
3. Error analysis
4. Fairness check and the final scoreboard

## 0. Setup

In [ ]:
# PURPOSE: get my repo and the dataset into Colab (skip this cell on my laptop).
# HOW: clone the repo, then the same download cell as Session 9.

import os, urllib.request, zipfile
from pathlib import Path

!git clone -q https://github.com/sopipi180/ComputerVisionandSpeechRecognition.git
%cd ComputerVisionandSpeechRecognition

DEST = Path("/content/milk10k")
if not (DEST / "metadata.csv").exists():
    urllib.request.urlretrieve(
        "https://isic-archive.s3.amazonaws.com/dois/10.34970-648456/milk10k.zip",
        "/content/milk10k.zip")
    with zipfile.ZipFile("/content/milk10k.zip") as z:
        z.extractall(DEST)

os.environ["MILK10K_DIR"] = str(DEST)      # common/config.py reads this

In [ ]:
# PURPOSE: libraries, my repo's config, and CPU or GPU.

import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import f1_score, ConfusionMatrixDisplay
from torchvision.models import resnet18, ResNet18_Weights, vit_b_16, ViT_B_16_Weights

# My repo: walk up until common/config.py is found (as in Sessions 6 and 7)
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / "common" / "config.py").is_file())
sys.path.insert(0, str(ROOT))
from common import config

CACHE = ROOT / "m4_best_model" / "_cache"      # decoded images and features (not in git)
CACHE.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(config.SEED)
print(device)

## 1. Data — my split, as in Milestones 2 and 3

The split CSVs are grouped by lesion, so both photos of a lesion stay on the same side.
One dermoscopic image per lesion, as in Sessions 5–7.

In [ ]:
# PURPOSE: load MY split and turn the labels into numbers.

CLASSES = config.CLASSES_3                     # ["Benign", "Indeterminate", "Malignant"]

splits = {}
for name in ["train", "val", "test"]:
    df = pd.read_csv(config.SPLIT_DIR / f"{name}.csv")
    splits[name] = df[df["image_type"].str.startswith("derm")].reset_index(drop=True)

y = {name: torch.tensor(df["diagnosis_1"].map(CLASSES.index).to_numpy()) for name, df in splits.items()}

for name, df in splits.items():
    print(name, len(df), df["diagnosis_1"].value_counts().to_dict())     # 3667 / 786 / 787

In [ ]:
# PURPOSE: decode every image once at 224x224 and keep it (Session 7 / Session 9).
# HOW: if the saved file exists, load it; if not, build it image by image and save it.

def load_images(df, name):
    f = CACHE / f"images_{name}.pt"
    if not f.exists():
        X = torch.stack([torch.from_numpy(np.array(Image.open(config.image_path(i)).convert("RGB").resize((224, 224))))
                         .permute(2, 0, 1) for i in df["isic_id"]])          # (H,W,3) -> (3,H,W)
        torch.save(X, f)
    return torch.load(f)

X = {name: load_images(df, name) for name, df in splits.items()}
print({name: tuple(x.shape) for name, x in X.items()})

In [ ]:
# PURPOSE: scale the pixels like ImageNet, because both networks learned on ImageNet (Session 7).
# HOW: uint8 -> float in [0, 1] -> subtract the mean and divide by the std, per channel, on the GPU.

MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

def prep(xb):
    return (xb.to(device).float() / 255 - MEAN) / STD

## 2. ResNet-18 vs ViT-B/16, a fair comparison

Session 9's recipe: both backbones **frozen**, every lesion turned into one vector once
(ResNet-18: 512 numbers, ViT: 768), then the **same** head trained on top.
Only the backbone changes. This time on all 3,667 training lesions.

In [ ]:
# PURPOSE: turn every lesion into one vector per backbone (Session 9, run on the GPU).
# HOW: remove each network's last layer, run all images once without gradients, save the result.

def backbone(name):
    if name == "resnet18":
        net = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        net.fc = nn.Identity()                    # output: 512 numbers instead of 1000 classes
    else:
        net = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)
        net.heads = nn.Identity()                 # output: the class token, 768 numbers
    return net.eval().to(device)

def features(name, X, split):
    f = CACHE / f"features_{name}_{split}.pt"
    if not f.exists():
        net, t0 = backbone(name), time.perf_counter()
        with torch.no_grad():
            out = torch.cat([net(prep(X[i:i + 64])).cpu() for i in range(0, len(X), 64)])
        torch.save(out, f)
        print(f"{name} {split}: {tuple(out.shape)} in {time.perf_counter() - t0:.0f} s")
    return torch.load(f)

feats = {name: {split: features(name, X[split], split) for split in ["train", "val", "test"]}
         for name in ["resnet18", "vit_b_16"]}

In [ ]:
# PURPOSE: the same head for both backbones (my Session 9 Exercise 4).

def train_head(F_train, y, n_classes=3, epochs=25):
    torch.manual_seed(0)
    head = nn.Linear(F_train.shape[1], n_classes)                      # the only weights we train
    class_weights = len(y) / (n_classes * torch.bincount(y, minlength=n_classes).float())
    loss_fn = nn.CrossEntropyLoss(weight=class_weights)                # rare classes count more (Session 6)
    optimizer = torch.optim.Adam(head.parameters(), lr=1e-3)
    for epoch in range(epochs):
        order = torch.randperm(len(y))                                 # new random order in each epoch
        for i in range(0, len(y), 64):
            idx = order[i:i + 64]                                      # one batch of 64 lesions
            optimizer.zero_grad()
            loss = loss_fn(head(F_train[idx]), y[idx])
            loss.backward()
            optimizer.step()
    return head

def macro_f1(head, F_, y):
    with torch.no_grad():
        pred = head(F_).argmax(1)
    return f1_score(y, pred, average="macro")

In [ ]:
# PURPOSE: train both heads and score them on the same validation lesions. Test stays closed.

heads, val_f1 = {}, {}
for name in ["resnet18", "vit_b_16"]:
    heads[name] = train_head(feats[name]["train"], y["train"])
    val_f1[name] = macro_f1(heads[name], feats[name]["val"], y["val"])
    print(f"{name:9s} validation macro-F1 = {val_f1[name]:.3f}")

In [ ]:
# PURPOSE: where does each backbone make its mistakes? (Session 9, section 4.2)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, name in zip(axes, ["resnet18", "vit_b_16"]):
    with torch.no_grad():
        pred = heads[name](feats[name]["val"]).argmax(1)
    ConfusionMatrixDisplay.from_predictions(y["val"], pred, display_labels=CLASSES, cmap="Oranges", ax=ax, colorbar=False)
    ax.set_title(f"{name}: macro-F1 {val_f1[name]:.3f}")
plt.tight_layout()

## 3. The final round: both fine-tuned

Session 7's recipe for both: unfreeze the **late** layers (they hold ImageNet-specific shapes) and keep
the early ones frozen (edges, colours and textures transfer fine). 6 epochs, lr 1e-4, flips, and keep
the best epoch on validation (early stopping).

- **ResNet-18:** `layer3` + `layer4` + a new head (my Session 7 mini challenge, 0.642)
- **ViT-B/16:** the last 2 of its 12 encoder blocks + a new head (the same idea for a transformer)

In [ ]:
# PURPOSE: add variety to the training images without changing their labels (Session 7).
# HOW: each image has a 50% chance of a left-right flip and, separately, of an up-down flip.

def augment(xb):
    n = xb.size(0)
    flip_h = torch.rand(n, 1, 1, 1, device=xb.device) < 0.5
    xb = torch.where(flip_h, xb.flip(3), xb)                 # left-right
    flip_v = torch.rand(n, 1, 1, 1, device=xb.device) < 0.5
    return torch.where(flip_v, xb.flip(2), xb)               # up-down

In [ ]:
# PURPOSE: one function to predict and one to train (Session 7), used by both models.
# HOW: Session 6's training step on batches of images. After each epoch, score on validation
#      and remember the best weights; at the end, go back to them (early stopping).

import copy

class_weights = (len(y["train"]) / (3 * torch.bincount(y["train"]).float())).to(device)

@torch.no_grad()
def predict(model, X, batch_size=128):
    model.eval()
    return torch.cat([model(prep(X[i:i + batch_size])).argmax(1).cpu() for i in range(0, len(X), batch_size)])

def fit(model, params, epochs, lr, batch_size=64):
    model.to(device)
    optimizer = torch.optim.Adam(params, lr=lr)              # params: only the weights we want to train
    loss_fn = nn.CrossEntropyLoss(weight=class_weights)
    best = {"f1": -1}
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(X["train"]))              # new random order in each epoch
        for i in range(0, len(X["train"]), batch_size):
            idx = order[i:i + batch_size]
            xb, yb = augment(prep(X["train"][idx])), y["train"][idx].to(device)
            optimizer.zero_grad()                            # forget the old gradients
            loss = loss_fn(model(xb), yb)                    # forward + loss
            loss.backward()                                  # backprop: compute the gradients
            optimizer.step()                                 # update the weights
        f1 = f1_score(y["val"], predict(model, X["val"]), average="macro")
        print(f"epoch {epoch}  val macro-F1 {f1:.3f}")
        if f1 > best["f1"]:
            best = {"f1": f1, "epoch": epoch, "state": copy.deepcopy(model.state_dict())}
    model.load_state_dict(best["state"])                     # early stopping: keep the best epoch
    return best

In [ ]:
# PURPOSE: ResNet-18 with layer3 + layer4 + a new head trainable (my Session 7 mini challenge).

torch.manual_seed(0)
resnet = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
resnet.fc = nn.Linear(512, 3)                                # new head: 3 classes
for name, p in resnet.named_parameters():
    p.requires_grad = name.startswith(("layer3", "layer4", "fc"))

trainable = [p for p in resnet.parameters() if p.requires_grad]
print(f"training {sum(p.numel() for p in trainable):,} of {sum(p.numel() for p in resnet.parameters()):,} weights")
best_resnet = fit(resnet, trainable, epochs=6, lr=1e-4)

In [ ]:
# PURPOSE: the same idea for ViT-B/16: its last 2 encoder blocks + a new head trainable.
# HOW: a weight's name says where it sits, e.g. "encoder.layers.encoder_layer_11.mlp.0.weight".
#      "encoder.ln" is the normalisation right after the last block, so it trains with them.

torch.manual_seed(0)
vit = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)
vit.heads = nn.Linear(768, 3)                                # new head: 3 classes
for name, p in vit.named_parameters():
    p.requires_grad = name.startswith(("encoder.layers.encoder_layer_10", "encoder.layers.encoder_layer_11",
                                       "encoder.ln", "heads"))

trainable = [p for p in vit.parameters() if p.requires_grad]
print(f"training {sum(p.numel() for p in trainable):,} of {sum(p.numel() for p in vit.parameters()):,} weights")
best_vit = fit(vit, trainable, epochs=6, lr=1e-4)

In [ ]:
# PURPOSE: choose the final model on VALIDATION only. The test set is still closed.

if best_vit["f1"] > best_resnet["f1"]:
    best_name, model = "ViT-B/16 fine-tuned", vit
else:
    best_name, model = "ResNet-18 fine-tuned", resnet
print(f"ResNet-18 fine-tuned {best_resnet['f1']:.3f} | ViT-B/16 fine-tuned {best_vit['f1']:.3f}")
print("final model:", best_name)

In [ ]:
# PURPOSE: open the test set ONCE, now that the choice is made, and score every M4 model for the scoreboard.

test_pred = {"ResNet-18 fine-tuned": predict(resnet, X["test"]),
             "ViT-B/16 fine-tuned": predict(vit, X["test"])}

m4 = pd.DataFrame([
    {"model": "ResNet-18 frozen + head", "val": val_f1["resnet18"],
     "test": macro_f1(heads["resnet18"], feats["resnet18"]["test"], y["test"])},
    {"model": "ViT-B/16 frozen + head", "val": val_f1["vit_b_16"],
     "test": macro_f1(heads["vit_b_16"], feats["vit_b_16"]["test"], y["test"])},
    {"model": "ResNet-18 fine-tuned", "val": best_resnet["f1"],
     "test": f1_score(y["test"], test_pred["ResNet-18 fine-tuned"], average="macro")},
    {"model": "ViT-B/16 fine-tuned", "val": best_vit["f1"],
     "test": f1_score(y["test"], test_pred["ViT-B/16 fine-tuned"], average="macro")},
]).round(3)
m4

In [ ]:
# PURPOSE: save the M4 deliverables: the final model and its predictions on my test split.

torch.save(model.state_dict(), ROOT / "m4_best_model" / "model_m4.pt")

out = splits["test"][["isic_id", "lesion_id", "diagnosis_1"]].copy()
out["predicted"] = [CLASSES[i] for i in test_pred[best_name]]
out.to_csv(ROOT / "m4_best_model" / "test_predictions.csv", index=False)
print(out["predicted"].value_counts().to_dict())

## 4. Error analysis — what the final model gets wrong

The test set was opened once, in section 3, and nothing below changes the model.
From here on I only **look** at its mistakes: which classes, which diagnoses behind them, and what the images look like.

In [ ]:
# PURPOSE: the final model's mistakes on test, as counts and per class.

from sklearn.metrics import classification_report

pred = test_pred[best_name]

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y["test"], pred, display_labels=CLASSES, cmap="Oranges", ax=ax, colorbar=False)
ax.set_title(f"{best_name} on test")
plt.tight_layout()
plt.show()

print(classification_report(y["test"], pred, target_names=CLASSES, digits=3))

In [ ]:
# PURPOSE: which diagnoses hide behind the mistakes?
# HOW: diagnosis_1 groups 11 diagnoses into 3 classes; the dx11 column says which one each lesion really is.

res = splits["test"][["isic_id", "diagnosis_1", "dx11", "sex", "age_approx", "anatom_site_general"]].copy()
res["predicted"] = [CLASSES[i] for i in pred]
res["wrong"] = res["diagnosis_1"] != res["predicted"]

by_dx = (res.groupby(["diagnosis_1", "dx11"])
            .agg(lesions=("wrong", "size"), error_rate=("wrong", "mean"))
            .round(2))
display(by_dx)

pd.crosstab(res["dx11"], res["predicted"])          # rows: real diagnosis, columns: what the model said

**What the tables show**

- **The costly mistake is missing a cancer**: 77 malignant lesions called Benign. They are not spread evenly.
  **Melanoma (MEL): 35 of 63 called Benign (56 %)**, against only 31 of 401 basal cell carcinomas (8 %).
  The deadliest skin cancer is the one the model misses most.
- A likely reason: melanoma and nevi (NV, moles) are both brown, pigmented, melanocytic lesions, and NV is the
  biggest benign group. The model seems to have learned "brown and pigmented = benign". BCC is 71 % of the
  Malignant class, so the model's idea of "malignant" is mostly "looks like a BCC".
- **The opposite mistake**: seborrhoeic keratoses (BKL) are benign, but 44 of 77 were called Malignant.
  They are rough and crusty, like the keratinocyte cancers (BCC, SCCKA).
- **Indeterminate is the same diagnosis as part of Malignant**: all 17 Indeterminate lesions are AKIEC, and
  27 Malignant lesions are AKIEC too. Whether an AKIEC is "Indeterminate" or "Malignant" is decided by the
  pathologist under the microscope, so the photo may simply not hold the answer. That caps the Indeterminate F1 (0.23).

In [ ]:
# PURPOSE: look at the worst mistakes, the ones the model was most sure about.
# HOW: softmax turns the 3 outputs into probabilities. Sort each kind of mistake by how confident the model was.

@torch.no_grad()
def predict_proba(model, X, batch_size=128):
    model.eval()
    return torch.cat([model(prep(X[i:i + batch_size])).softmax(1).cpu() for i in range(0, len(X), batch_size)])

proba = predict_proba(model, X["test"])
for c in CLASSES:
    res[f"p_{c}"] = proba[:, CLASSES.index(c)].numpy()

from common import viz
viz.SHOW = True                                       # show the figure in the notebook as well as saving it
FIG = ROOT / "m4_best_model" / "figures"

def worst(true, said, n=12):
    """The n lesions of class `true` called `said`, most confident first. res's index = position in X["test"]."""
    w = res[(res["diagnosis_1"] == true) & (res["predicted"] == said)].sort_values(f"p_{said}", ascending=False).head(n)
    viz.show_grid([X["test"][i] for i in w.index],
                  titles=[f"{d}  P={p:.2f}" for d, p in zip(w["dx11"], w[f"p_{said}"])],
                  ncols=6, title=f"{true} lesions called {said}, most confident first",
                  filename=f"m4_{true}_called_{said}.png".lower(), fig_dir=FIG)

worst("Malignant", "Benign")                          # missed cancers: the mistake that matters most
worst("Benign", "Malignant")                          # false alarms: an unnecessary biopsy

**What I see in the pictures** — *(write 2–3 sentences after looking at the two grids: Are the missed
cancers mostly melanomas? Do they look like ordinary moles? Are there hairs, rulers, dark corners or blur?
Do the false alarms look crusty or ulcerated?)*

## 5. Fairness check — does the model work equally well for everyone?

The metadata gives three ways to split the patients: **sex**, **age** and **body site**.
For each group: how many lesions, how many of them are malignant, macro-F1, and **malignant recall**
(the share of cancers the model catches — the number a doctor cares about at triage).
Groups under 30 lesions are skipped: too few to say anything.

In [ ]:
# PURPOSE: the same metrics, one patient group at a time.

from sklearn.metrics import recall_score

res["age_group"] = pd.cut(res["age_approx"], [0, 49, 69, 120], labels=["under 50", "50-69", "70+"])   # 4 lesions have no age
res["site"] = res["anatom_site_general"].fillna("unknown")

rows = []
for col in ["sex", "age_group", "site"]:
    for group, g in res.groupby(col, observed=True):
        if len(g) < 30:
            continue
        rows.append({"by": col, "group": group, "lesions": len(g),
                     "% malignant": round(100 * (g["diagnosis_1"] == "Malignant").mean()),
                     "macro-F1": round(f1_score(g["diagnosis_1"], g["predicted"], average="macro"), 3),
                     "malignant recall": round(recall_score(g["diagnosis_1"] == "Malignant", g["predicted"] == "Malignant"), 3)})

fairness = pd.DataFrame(rows)
fairness.to_csv(ROOT / "m4_best_model" / "fairness.csv", index=False)
fairness

In [ ]:
# PURPOSE: why is the under-50 group worse? Look at which diagnoses it contains and what the model said.

young = res[res["age_group"] == "under 50"]
pd.crosstab(young["dx11"], young["predicted"])

**What the fairness check shows**

- **Age is the biggest gap.** Malignant recall is about 0.61 for patients under 50, against 0.87–0.88 for 50+.
  The crosstab explains it: **13 of the 14 melanomas in under-50s were called Benign**. Young patients' cancers
  are more often melanoma, and melanoma is exactly what the model misses (section 4).
- **Sex: a smaller gap** (recall 0.80 for women, 0.87 for men). Same cause: women in the test set have
  relatively more melanomas (27 of 317 lesions vs 36 of 470).
- **Body site**: recall is between 0.80 and 0.90 for every site; 302 lesions have no site recorded.
- So the "fairness" gaps are mostly **one problem — melanoma** — showing up as an age and sex gap.
  Re-weighting age groups would not fix it; more melanoma examples (or the 11-class target) might.
- **Limits**: only 14 melanomas in under-50s, so these numbers move a lot with a few lesions.
  The metadata has **no skin tone**, so the most important fairness question in dermatology (does it work on dark
  skin?) cannot be tested here.

## 6. Final scoreboard

Earlier rows are copied from the README (Milestones 2 and 3); the M4 rows come from section 3.
Same split, same dermoscopic photo, macro-F1. Validation chose, test was scored once.

In [ ]:
# PURPOSE: the whole project in one table, saved next to the model.

earlier = pd.DataFrame([
    ("5", "majority (always Malignant)",                0.272, 0.278),
    ("5", "random forest, 24 histogram features",       0.511, 0.525),
    ("6", "MLP 33-64-3, handcrafted features",          0.463, 0.483),
    ("6", "MLP 3072-64-3, raw 32x32 pixels",            0.524, 0.456),
    ("7", "CNN from scratch",                           0.520, None),
    ("7", "ResNet-18 frozen + linear head",             0.563, None),
    ("7", "ResNet-18, layer4 fine-tuned",               0.638, None),
    ("7", "ResNet-18, layer3 + layer4 fine-tuned",      0.642, 0.590),
    ("7", "ResNet-18, all layers fine-tuned",           0.529, None),
], columns=["session", "model", "val", "test"])

board = pd.concat([earlier, m4.assign(session="M4")[["session", "model", "val", "test"]]], ignore_index=True)
board.to_csv(ROOT / "m4_best_model" / "scoreboard.csv", index=False)
board

**Conclusion**

- **Final model: ViT-B/16 with its last 2 blocks fine-tuned** — chosen on validation (0.663), **test 0.589**.
- On test it **ties** the Milestone 3 ResNet-18 (0.590) and the frozen ViT (0.589). On 787 test lesions,
  of which only 17 are Indeterminate, differences of 0.001–0.01 are noise.
- The same ResNet-18 recipe scored 0.590 in M3 and 0.582 here: that gap is roughly the run-to-run noise.
- What M4 adds is not a higher number but an understanding of the errors: **the model misses melanoma**,
  and that is also why it does worse for patients under 50. That is the first thing to fix next.

In [ ]:
from google.colab import files
for f in ["model_m4.pt", "test_predictions.csv", "fairness.csv", "scoreboard.csv",
          "figures/m4_malignant_called_benign.png", "figures/m4_benign_called_malignant.png"]:
    files.download(str(ROOT / "m4_best_model" / f))
